In [7]:
from pyspark.sql import SparkSession
import getpass 
username=getpass.getuser()
spark=SparkSession. \
    builder. \
    config('spark.ui.port','0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    master('yarn'). \
    getOrCreate()

## Associating points to the grades in order to calculate the Loan Score

In [8]:
spark.conf.set("spark.sql.unacceptable_rated_pts", 0)
spark.conf.set("spark.sql.very_bad_rated_pts", 100)
spark.conf.set("spark.sql.bad_rated_pts", 250)
spark.conf.set("spark.sql.good_rated_pts", 500)
spark.conf.set("spark.sql.very_good_rated_pts", 650)
spark.conf.set("spark.sql.excellent_rated_pts", 800)

In [9]:
spark.conf.set("spark.sql.unacceptable_grade_pts", 750)
spark.conf.set("spark.sql.very_bad_grade_pts", 1000)
spark.conf.set("spark.sql.bad_grade_pts", 1500)
spark.conf.set("spark.sql.good_grade_pts", 2000)
spark.conf.set("spark.sql.very_good_grade_pts", 2500)

## The tables required to calculate the Loan Score

customers_new 

loans

loans_repayments

loans_defaulters_delinq_new

loans_defaulters_detail_red_enq_new

In [10]:
bad_customer_data_final_df = spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("/user/itv017907/lendingclub/bad/bad_customer_data_final")

In [11]:
bad_customer_data_final_df

member_id
df51e1f6c20da2046...
981a0200552d3e4c1...
b74a51c1f9f7db72f...
48f3a3908b7ef5592...
03f71ce106a7d33b3...
73a5bbf6f0ba325d8...
a596e2006964267d6...
104f6a6b44aa47c15...
066ddaa64bee66dff...
d4c67dd352c3e6fc9...


In [12]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [13]:
ph_df = spark.sql("select c.member_id, \
   case \
   when p.last_payment_amount < (c.monthly_installment * 0.5) then ${spark.sql.very_bad_rated_pts} \
   when p.last_payment_amount >= (c.monthly_installment * 0.5) and p.last_payment_amount < c.monthly_installment then ${spark.sql.bad_rated_pts} \
   when (p.last_payment_amount = (c.monthly_installment)) then ${spark.sql.good_rated_pts} \
   when p.last_payment_amount > (c.monthly_installment) and p.last_payment_amount <= (c.monthly_installment * 1.50) then ${spark.sql.very_good_rated_pts} \
   when p.last_payment_amount > (c.monthly_installment * 1.50) then ${spark.sql.excellent_rated_pts} \
   else ${spark.sql.unacceptable_rated_pts} \
   end as last_payment_pts, \
   case \
   when p.total_payment>= (c.funded_amount * 0.50) then ${spark.sql.very_good_rated_pts} \
   when p.total_payment < (c.funded_amount * 0.50) and p.total_payment > 0 then ${spark.sql.good_rated_pts} \
   when p.total_payment = 0 or (p.total_payment) is null then ${spark.sql.unacceptable_rated_pts} \
   end as total_payment_pts \
from itv017907_lending_club.loans_repayments p \
inner join itv017907_lending_club.loans c on c.loan_id = p.loan_id where member_id NOT IN (select member_id from bad_data_customer)")

In [14]:
# Check what loan_ids actually look like in each table
spark.sql("SELECT loan_id FROM itv017907_lending_club.loans LIMIT 5").show()
spark.sql("SELECT loan_id FROM itv017907_lending_club.loans_repayments LIMIT 5").show()

# Check if ANY loan_ids overlap at all
spark.sql("""
SELECT COUNT(*) FROM itv017907_lending_club.loans_repayments r
INNER JOIN itv017907_lending_club.loans l ON l.loan_id = r.loan_id
""").show()

+--------+
| loan_id|
+--------+
|68407277|
|68355089|
|68341763|
|66310712|
|68476807|
+--------+

+-------+
|loan_id|
+-------+
| 491699|
| 491685|
| 491667|
| 491160|
| 491675|
+-------+

+--------+
|count(1)|
+--------+
| 2259549|
+--------+



In [15]:
spark.sql("""
SELECT COUNT(*) 
FROM bad_data_customer 
WHERE member_id IS NULL
""").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



In [16]:
spark.sql("SELECT COUNT(*) FROM itv017907_lending_club.loans").show()

+--------+
|count(1)|
+--------+
| 2260667|
+--------+



In [17]:
spark.sql("SELECT COUNT(*) FROM itv017907_lending_club.loans_repayments").show()

+--------+
|count(1)|
+--------+
| 2259549|
+--------+



In [18]:
ph_df

member_id,last_payment_pts,total_payment_pts
dcec9334e70f1cc95...,800,650
fc58ca61f51f9dcac...,500,650
2fb62a6ca51063b11...,500,650
488268a5531951622...,800,650
ade6026208e48f5f9...,500,650
7c8b0ca6acddfaeb1...,800,650
a707b7fe7c38bad65...,800,650
1df639cddea30c288...,800,650
22d67005e12d8726d...,500,650
009cf312bd46551b4...,500,650


In [19]:
ph_df.createOrReplaceTempView("ph_pts")

In [20]:
spark.sql("select * from ph_pts")

member_id,last_payment_pts,total_payment_pts
dcec9334e70f1cc95...,800,650
fc58ca61f51f9dcac...,500,650
2fb62a6ca51063b11...,500,650
488268a5531951622...,800,650
ade6026208e48f5f9...,500,650
7c8b0ca6acddfaeb1...,800,650
a707b7fe7c38bad65...,800,650
1df639cddea30c288...,800,650
22d67005e12d8726d...,500,650
009cf312bd46551b4...,500,650


## Loan Score Calculation Criteria 2: Loan Defaulters History(ldh)

In [21]:
ldh_ph_df = spark.sql(
    "select p.*, \
    CASE \
    WHEN d.delinq_2yrs = 0 THEN ${spark.sql.excellent_rated_pts} \
    WHEN d.delinq_2yrs BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
    WHEN d.delinq_2yrs BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
    WHEN d.delinq_2yrs > 5 OR d.delinq_2yrs IS NULL THEN ${spark.sql.unacceptable_grade_pts} \
    END AS delinq_pts, \
    CASE \
    WHEN l.pub_rec = 0 THEN ${spark.sql.excellent_rated_pts} \
    WHEN l.pub_rec BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
    WHEN l.pub_rec BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
    WHEN l.pub_rec > 5 OR l.pub_rec IS NULL THEN ${spark.sql.very_bad_rated_pts} \
    END AS public_records_pts, \
    CASE \
    WHEN l.pub_rec_bankruptcies = 0 THEN ${spark.sql.excellent_rated_pts} \
    WHEN l.pub_rec_bankruptcies BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
    WHEN l.pub_rec_bankruptcies BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
    WHEN l.pub_rec_bankruptcies > 5 OR l.pub_rec_bankruptcies IS NULL THEN ${spark.sql.very_bad_rated_pts} \
    END as public_bankruptcies_pts, \
    CASE \
    WHEN l.inq_last_6mths = 0 THEN ${spark.sql.excellent_rated_pts} \
    WHEN l.inq_last_6mths BETWEEN 1 AND 2 THEN ${spark.sql.bad_rated_pts} \
    WHEN l.inq_last_6mths BETWEEN 3 AND 5 THEN ${spark.sql.very_bad_rated_pts} \
    WHEN l.inq_last_6mths > 5 OR l.inq_last_6mths IS NULL THEN ${spark.sql.unacceptable_rated_pts} \
    END AS enq_pts \
    FROM itv017907_lending_club.loans_defaulters_detail_rec_enq_new l \
    INNER JOIN itv017907_lending_club.loans_defaulters_delinq_new d ON d.member_id = l.member_id  \
    INNER JOIN ph_pts p ON p.member_id = l.member_id where l.member_id NOT IN (select member_id from bad_data_customer)")

In [22]:
ldh_ph_df.createOrReplaceTempView("ldh_ph_pts")

In [23]:
spark.sql("select * from ldh_ph_pts")

member_id,last_payment_pts,total_payment_pts,delinq_pts,public_records_pts,public_bankruptcies_pts,enq_pts
00151ece27c7ca280...,800,650,800,800,800,250
003e1e6cbd2920bbb...,500,650,250,250,250,800
005b4c3db3fce07dc...,500,650,250,250,800,250
008515010997a80e1...,500,650,800,800,800,250
00d10706191726593...,800,650,800,800,800,250
00da692a960ef39f1...,800,650,800,800,800,250
00ec12e214988d8b3...,650,650,800,800,800,250
00fc8144cb210ba8c...,500,650,250,250,250,800
0110b7f5f4010cbaf...,800,650,800,800,800,250
01270634dc82d1645...,800,650,800,800,800,250


## Criteria 3:Financial Health (fh)

In [24]:
fh_ldh_ph_df = spark.sql("select ldef.*, \
   CASE \
   WHEN LOWER(l.loan_status) LIKE '%fully paid%' THEN ${spark.sql.excellent_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%current%' THEN ${spark.sql.good_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%in grace period%' THEN ${spark.sql.bad_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%late (16-30 days)%' OR LOWER(l.loan_status) LIKE '%late (31-120 days)%' THEN ${spark.sql.very_bad_rated_pts} \
   WHEN LOWER(l.loan_status) LIKE '%charged off%' THEN ${spark.sql.unacceptable_rated_pts} \
   else ${spark.sql.unacceptable_rated_pts} \
   END AS loan_status_pts, \
   CASE \
   WHEN LOWER(a.home_ownership) LIKE '%own' THEN ${spark.sql.excellent_rated_pts} \
   WHEN LOWER(a.home_ownership) LIKE '%rent' THEN ${spark.sql.good_rated_pts} \
   WHEN LOWER(a.home_ownership) LIKE '%mortgage' THEN ${spark.sql.bad_rated_pts} \
   WHEN LOWER(a.home_ownership) LIKE '%any' OR LOWER(a.home_ownership) IS NULL THEN ${spark.sql.very_bad_rated_pts} \
   END AS home_pts, \
   CASE \
   WHEN l.funded_amount <= (a.total_high_credit_limit * 0.10) THEN ${spark.sql.excellent_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.10) AND l.funded_amount <= (a.total_high_credit_limit * 0.20) THEN ${spark.sql.very_good_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.20) AND l.funded_amount <= (a.total_high_credit_limit * 0.30) THEN ${spark.sql.good_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.30) AND l.funded_amount <= (a.total_high_credit_limit * 0.50) THEN ${spark.sql.bad_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.50) AND l.funded_amount <= (a.total_high_credit_limit * 0.70) THEN ${spark.sql.very_bad_rated_pts} \
   WHEN l.funded_amount > (a.total_high_credit_limit * 0.70) THEN ${spark.sql.unacceptable_rated_pts} \
   else ${spark.sql.unacceptable_rated_pts} \
   END AS credit_limit_pts, \
   CASE \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A1' THEN ${spark.sql.excellent_rated_pts} \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A2' THEN (${spark.sql.excellent_rated_pts} * 0.95) \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A3' THEN (${spark.sql.excellent_rated_pts} * 0.90) \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A4' THEN (${spark.sql.excellent_rated_pts} * 0.85) \
   WHEN (a.grade) = 'A' and (a.sub_grade)='A5' THEN (${spark.sql.excellent_rated_pts} * 0.80) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B1' THEN (${spark.sql.very_good_rated_pts}) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B2' THEN (${spark.sql.very_good_rated_pts} * 0.95) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B3' THEN (${spark.sql.very_good_rated_pts} * 0.90) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B4' THEN (${spark.sql.very_good_rated_pts} * 0.85) \
   WHEN (a.grade) = 'B' and (a.sub_grade)='B5' THEN (${spark.sql.very_good_rated_pts} * 0.80) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C1' THEN (${spark.sql.good_rated_pts}) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C2' THEN (${spark.sql.good_rated_pts} * 0.95) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C3' THEN (${spark.sql.good_rated_pts} * 0.90) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C4' THEN (${spark.sql.good_rated_pts} * 0.85) \
   WHEN (a.grade) = 'C' and (a.sub_grade)='C5' THEN (${spark.sql.good_rated_pts} * 0.80) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D1' THEN (${spark.sql.bad_rated_pts}) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D2' THEN (${spark.sql.bad_rated_pts} * 0.95) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D3' THEN (${spark.sql.bad_rated_pts} * 0.90) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D4' THEN (${spark.sql.bad_rated_pts} * 0.85) \
   WHEN (a.grade) = 'D' and (a.sub_grade)='D5' THEN (${spark.sql.bad_rated_pts} * 0.80) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E1' THEN (${spark.sql.very_bad_rated_pts}) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E2' THEN (${spark.sql.very_bad_rated_pts} * 0.95) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E3' THEN (${spark.sql.very_bad_rated_pts} * 0.90) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E4' THEN (${spark.sql.very_bad_rated_pts} * 0.85) \
   WHEN (a.grade) = 'E' and (a.sub_grade)='E5' THEN (${spark.sql.very_bad_rated_pts} * 0.80) \
   WHEN (a.grade) in ('F', 'G') THEN (${spark.sql.unacceptable_rated_pts}) \
   END AS grade_pts \
   FROM ldh_ph_pts ldef \
   INNER JOIN itv017907_lending_club.loans l ON ldef.member_id = l.member_id \
   INNER JOIN itv017907_lending_club.customers_new a ON a.member_id = ldef.member_id where ldef.member_id NOT IN (select member_id from bad_data_customer)") 

In [25]:
fh_ldh_ph_df.createOrReplaceTempView("fh_ldh_ph_pts")

In [26]:
spark.sql("select * from fh_ldh_ph_pts")

member_id,last_payment_pts,total_payment_pts,delinq_pts,public_records_pts,public_bankruptcies_pts,enq_pts,loan_status_pts,home_pts,credit_limit_pts,grade_pts
00151ece27c7ca280...,800,650,800,800,800,250,800,500,250,500.00
003e1e6cbd2920bbb...,500,650,250,250,250,800,500,250,800,640.00
005b4c3db3fce07dc...,500,650,250,250,800,250,500,250,500,520.00
008515010997a80e1...,500,650,800,800,800,250,500,500,500,250.00
00d10706191726593...,800,650,800,800,800,250,800,500,250,425.00
00da692a960ef39f1...,800,650,800,800,800,250,800,800,500,0.00
00ec12e214988d8b3...,650,650,800,800,800,250,800,500,650,680.00
00fc8144cb210ba8c...,500,650,250,250,250,800,500,250,800,500.00
0110b7f5f4010cbaf...,800,650,800,800,800,250,800,250,800,450.00
01270634dc82d1645...,800,650,800,800,800,250,800,250,800,400.00


#### Final loan score calculation by considering all the 3 criterias with the following %**

#### 1. Payment History = 20%
#### 2. Loan Defaults = 45%
#### 3. Financial Health = 35%

In [27]:
loan_score = spark.sql("SELECT member_id, \
((last_payment_pts+total_payment_pts)*0.20) as payment_history_pts, \
((delinq_pts + public_records_pts + public_bankruptcies_pts + enq_pts) * 0.45) as defaulters_history_pts, \
((loan_status_pts + home_pts + credit_limit_pts + grade_pts)*0.35) as financial_health_pts \
FROM fh_ldh_ph_pts")

In [28]:
loan_score

member_id,payment_history_pts,defaulters_history_pts,financial_health_pts
00151ece27c7ca280...,290.00,1192.50,717.5000
003e1e6cbd2920bbb...,230.00,697.50,766.5000
005b4c3db3fce07dc...,230.00,697.50,619.5000
008515010997a80e1...,230.00,1192.50,612.5000
00d10706191726593...,290.00,1192.50,691.2500
00da692a960ef39f1...,290.00,1192.50,735.0000
00ec12e214988d8b3...,260.00,1192.50,920.5000
00fc8144cb210ba8c...,230.00,697.50,717.5000
0110b7f5f4010cbaf...,290.00,1192.50,805.0000
01270634dc82d1645...,290.00,1192.50,787.5000


In [29]:
final_loan_score = loan_score.withColumn('loan_score', loan_score.payment_history_pts + loan_score.defaulters_history_pts + loan_score.financial_health_pts)

In [30]:
final_loan_score.createOrReplaceTempView("loan_score_eval")

In [31]:
loan_score_final = spark.sql("select ls.*, \
case \
WHEN loan_score > ${spark.sql.very_good_grade_pts} THEN 'A' \
WHEN loan_score <= ${spark.sql.very_good_grade_pts} AND loan_score > ${spark.sql.good_grade_pts} THEN 'B' \
WHEN loan_score <= ${spark.sql.good_grade_pts} AND loan_score > ${spark.sql.bad_grade_pts} THEN 'C' \
WHEN loan_score <= ${spark.sql.bad_grade_pts} AND loan_score  > ${spark.sql.very_bad_grade_pts} THEN 'D' \
WHEN loan_score <= ${spark.sql.very_bad_grade_pts} AND loan_score > ${spark.sql.unacceptable_grade_pts} THEN 'E'  \
WHEN loan_score <= ${spark.sql.unacceptable_grade_pts} THEN 'F' \
end as loan_final_grade \
from loan_score_eval ls")

In [32]:
loan_score_final.createOrReplaceTempView("loan_final_table")

In [33]:
spark.sql("select * from loan_final_table where loan_final_grade in ('C')")

member_id,payment_history_pts,defaulters_history_pts,financial_health_pts,loan_score,loan_final_grade
003e1e6cbd2920bbb...,230.00,697.50,766.5000,1694.0000,C
005b4c3db3fce07dc...,230.00,697.50,619.5000,1547.0000,C
00fc8144cb210ba8c...,230.00,697.50,717.5000,1645.0000,C
0134d807fbc9e8ff8...,230.00,1192.50,553.0000,1975.5000,C
013630bb77d0f3c6a...,290.00,1192.50,399.0000,1881.5000,C
017ce564dc0d6f975...,200.00,945.00,591.5000,1736.5000,C
01b48af926b98a718...,200.00,1192.50,573.1250,1965.6250,C
01d0c48835e969a01...,200.00,1192.50,262.5000,1655.0000,C
024f077b5b4e47ea0...,230.00,1192.50,437.5000,1860.0000,C
025ae8414d0b7147c...,230.00,945.00,572.2500,1747.2500,C


In [34]:
spark.sql("select count(*) from loan_final_table")

count(1)
536639


In [35]:
loan_score_final.write \
.format("parquet") \
.mode("overwrite") \
.save(f"/user/{username}/lendingclubproject/processed/loan_score")